In [1]:
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
SEQUENCE_LENGTH = 50
NUM_FEATURES = 18

In [3]:
current_dir = Path.cwd()
gesture_data_dir = current_dir.parent.parent / "data" / "gesture"
gesture_model_dir = current_dir.parent.parent / "models" / "gesture"

In [4]:
def resample_sequence(df: pd.DataFrame, target_length: int) -> pd.DataFrame:
        """
        Adjusts the number of rows in the DataFrame to target_length.
        If there are fewer rows, interpolation is performed.
        If there are more rows, points are selected uniformly.

        Args:
            df: Pandas DataFrame containing time-series data (one gesture).
            target_length: Desired number of points after resampling.

        Returns:
            Pandas DataFrame with the same structure and target_length rows.
        """
        df = df.reset_index(drop=True)
        current_length = len(df)

        if current_length < target_length:
            new_index = np.linspace(0, current_length - 1, target_length)
            df_resampled = df.reindex(new_index)
            df_resampled = df_resampled.interpolate(method="linear")
            return df_resampled.reset_index(drop=True)

        elif current_length > target_length:
            indices = np.linspace(0, current_length - 1, target_length, dtype=int)
            df_resampled = df.iloc[indices].reset_index(drop=True)
            return df_resampled

        else:
            return df

In [5]:
file = random.choice(os.listdir(gesture_data_dir))

if file.endswith(".csv"):
    label = file.split("_")[0]
    
    df = pd.read_csv(os.path.join(gesture_data_dir, file))
    df_resampled = resample_sequence(df, SEQUENCE_LENGTH)
    
    if df_resampled.shape != (SEQUENCE_LENGTH, NUM_FEATURES):
        print(f"Skipping {file} after resampling — got {df_resampled.shape}")
    
    sample = df_resampled.values.astype(float)

print(f"Loaded {file.title()} gesture.")

Loaded Good-Morning_1.Csv gesture.


In [6]:
# Load pipeline
model = tf.keras.models.load_model(gesture_model_dir / "model.keras")
scaler = joblib.load(gesture_model_dir / "scaler.pkl")
encoder = joblib.load(gesture_model_dir / "encoder.pkl")

# Convert to numpy
sample = np.asarray(sample, dtype=np.float32)

T, F = sample.shape

# Scale exactly as during training
sample_2d = sample.reshape(-1, F)

sample_scaled = scaler.transform(sample_2d)

# Restore LSTM shape: (1, T, F)
sample_scaled = sample_scaled.reshape(1, T, F)

# Prediction
probabilities = model.predict(
    sample_scaled,
    verbose=0
)[0]

# Get predicted class
predicted_index = np.argmax(probabilities)

# Get label
predicted_label = encoder.inverse_transform(
    np.eye(len(probabilities))[predicted_index].reshape(1, -1)
)[0][0]

# Confidence
confidence = float(probabilities[predicted_index])

print({
    "predictedLabel": str(predicted_label),
    "confidence": confidence,
})

{'predictedLabel': 'good-morning', 'confidence': 0.976088285446167}
